<a href="https://colab.research.google.com/github/Husan2/NTU_GAI/blob/main/0325%E6%89%93%E9%80%A0%E8%87%AA%E5%B7%B1%E7%9A%84%E5%B0%8D%E8%A9%B1%E6%A9%9F%E5%99%A8%E4%BA%BA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧙‍♀️用 Groq API 打造一個擁有哈利波特中，麥米奈娃教授（Minerva McGonagall）性格的對話機器人

本週作業的目標是透過 **Groq API** 打造一個具有明確人設的智慧對話機器人。我選擇了《哈利波特》系列中深具智慧與威嚴的角色──**麥米奈娃·麥教授（Minerva McGonagall）**，並嘗試讓她展現出溫柔、親切但堅定的教學風格。

在人設設計上，我使用 `system` prompt 明確設定教授的身份、個性與語氣，並要求她使用繁體中文回應，讓整體互動體驗更貼近霍格華茲的氛圍。

---
## 🪄 對話一：魔法與個性的關聯

我問教授：

> 教授，您覺得一個人會因為選擇的魔法而變成不同的人嗎？

她的回答令人印象深刻，不僅回應了問題，還加入了價值觀引導。她指出：**魔法的選擇反映的是人的價值觀與個性，但不會從根本改變一個人的本質**；真正定義一個人的是他如何使用這股力量。  
她也提醒我：選擇魔法的過程，其實也是一種成長與轉變。這讓我聯想到現實中我們對技術與知識的選擇，關鍵不在工具本身，而在於我們的使用方式。

---

## 📚 對話二：學習上的困難與鼓勵

我接著問教授：

> 我在變形術小考總是記不住那個魔法咒語的語調，有什麼練習方法嗎？

教授以非常細膩又具體的方式回應，分析了問題的可能原因，並給予多種練習策略：

- 重複誦讀魔法咒語，強化語調熟悉度  
- 將咒語與動作結合，加深記憶連結  
- 在不同情境下練習，以提高應變能力  

她最後以一句「**我們可以一起努力**」來鼓勵我，展現出既嚴謹又溫暖的教學態度，完全貼合原著角色形象。

---
## 🤔 心得

這次作業讓我體會到：**prompt 設計不只是技術操作，更是一種角色建構與風格掌握的藝術。**

Groq API 搭配 `llama3-70b-8192` 模型的回應品質穩定，風格呈現也十分自然。只要人設設定得當，模型便能準確模擬角色語氣與價值觀。

未來我希望挑戰更複雜的多輪對話與角色切換，例如讓使用者能夠「選擇教授」或「自訂角色」，打造更具沉浸感的互動體驗。

In [1]:
# Step 1: 安裝必要套件
!pip install openai gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 MB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.2/322.2 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 4.0 MB/s eta 0:00:00


In [2]:
# Step 2: 載入環境變數與 API 設定（你要先申請好 Groq API 並存到 userdata）
import os
from google.colab import userdata
from openai import OpenAI

In [3]:
# ✅ 直接在這裡設定你的 Groq API 金鑰
os.environ["OPENAI_API_KEY"] = "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"

# ✅ 使用 Groq API（注意 base_url）
client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
    base_url="https://api.groq.com/openai/v1"
)

In [7]:
# 人設設定：麥米奈娃教授風格
system = """
你是霍格華茲的變形術教授，麥米奈娃·麥教授（Minerva McGonagall）。
你的性格溫柔但堅定、智慧且親切，總是給予學生有原則的建議與暖心的鼓勵。
請你用這樣的語氣回答學生的問題：理性中帶著慈愛、親切但有威嚴。
請用繁體中文回答學生的提問。
"""

In [5]:
# 對話函數
def chat_with_mcgonagall(user_input, history=[]):
    messages = [{"role": "system", "content": system}]
    for u, a in history:
        messages.append({"role": "user", "content": u})
        messages.append({"role": "assistant", "content": a})
    messages.append({"role": "user", "content": user_input})

    response = client.chat.completions.create(
        model="llama3-70b-8192",  # ✅ Groq 支援的 LLaMA 3 模型
        messages=messages,
        temperature=0.7  # 稍微增加溫度以展現溫柔與個性
    )
    reply = response.choices[0].message.content
    history.append((user_input, reply))
    return reply, history

In [8]:
# Gradio 介面
import gradio as gr

with gr.Blocks() as demo:
    gr.Markdown("## 🧙‍♀️ 麥米奈娃教授・智慧對話機器人（Groq API）")
    chatbot = gr.Chatbot()
    msg = gr.Textbox(label="請向麥教授提問")
    state = gr.State([])

    def user_submit(user_message, history):
        reply, history = chat_with_mcgonagall(user_message, history)
        return history, history

    msg.submit(user_submit, [msg, state], [chatbot, state])

demo.launch()

<ipython-input-8-1093251fa61c>:6: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot()


Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c32f105c64f7f6e995.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
